# 09 — Artificial Neural Networks


The catalog includes scikit-learn MLP ANN models. The preprocessing and early-stopping logic are shared with the other leakage-safe pipelines.


In [1]:
#Imports

from pathlib import Path
import json
import os
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

ROOT = Path.cwd().parent
FEATURE_DIR = ROOT / "artifacts" / "features"
MODEL_DIR = ROOT / "artifacts" / "models"
PREDICTION_DIR = ROOT / "artifacts" / "predictions"
CHART_DIR = ROOT / "artifacts" / "charts"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
# Load Processed features

X_train = pd.read_parquet(
    FEATURE_DIR / "X_train_processed.parquet"
)

X_test = pd.read_parquet(
    FEATURE_DIR / "X_test_processed.parquet"
)

with open(
    FEATURE_DIR / "feature_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)

TARGET_COLUMN = feature_metadata["target_column"]
problem_type = feature_metadata["problem_type"]

y_train_original = pd.read_parquet(
    FEATURE_DIR / "y_train.parquet"
)[TARGET_COLUMN]

y_test_original = pd.read_parquet(
    FEATURE_DIR / "y_test.parquet"
)[TARGET_COLUMN]

X_train_array = X_train.to_numpy(dtype=np.float32)
X_test_array = X_test.to_numpy(dtype=np.float32)

print("Problem type:", problem_type)
print("Target:", TARGET_COLUMN)
print("Training shape:", X_train_array.shape)
print("Testing shape:", X_test_array.shape)
print("Input features:", X_train_array.shape[1])

Problem type: classification
Target: purchased
Training shape: (400, 8)
Testing shape: (100, 8)
Input features: 8


In [3]:
# Target Prepare

label_encoder = None
target_scaler = None
class_weight_dictionary = None

if problem_type == "classification":
    label_encoder = LabelEncoder()

    y_train_array = label_encoder.fit_transform(
        y_train_original
    )

    y_test_array = label_encoder.transform(
        y_test_original
    )

    number_of_classes = len(label_encoder.classes_)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_array),
        y=y_train_array,
    )

    class_weight_dictionary = {
        int(class_label): float(weight)
        for class_label, weight in zip(
            np.unique(y_train_array),
            class_weights,
        )
    }

    print("Number of classes:", number_of_classes)
    print("Classes:", label_encoder.classes_)
    print("Class weights:", class_weight_dictionary)

else:
    target_scaler = StandardScaler()

    y_train_array = target_scaler.fit_transform(
        y_train_original.to_numpy().reshape(-1, 1)
    ).ravel()

    y_test_array = target_scaler.transform(
        y_test_original.to_numpy().reshape(-1, 1)
    ).ravel()

    number_of_classes = None

    print("Regression target was standardized.")

Number of classes: 2
Classes: [0 1]
Class weights: {0: 1.2738853503184713, 1: 0.823045267489712}


In [4]:
#Creating internal Validation Data

stratify_values = None

if problem_type == "classification":
    stratify_values = y_train_array

(
    X_ann_train,
    X_ann_validation,
    y_ann_train,
    y_ann_validation,
) = train_test_split(
    X_train_array,
    y_train_array,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_values,
)

print("ANN training shape:", X_ann_train.shape)
print("ANN validation shape:", X_ann_validation.shape)
print("Final untouched test shape:", X_test_array.shape)

ANN training shape: (320, 8)
ANN validation shape: (80, 8)
Final untouched test shape: (100, 8)


In [5]:
#ANN Architecture

def build_ann_model(
    input_feature_count,
    problem_type,
    number_of_classes=None,
):
    inputs = keras.Input(
        shape=(input_feature_count,),
        name="input_features",
    )

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(0.0005),
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(0.0005),
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(
        32,
        activation="relu",
    )(x)

    if problem_type == "classification":
        if number_of_classes == 2:
            outputs = layers.Dense(
                1,
                activation="sigmoid",
                name="prediction",
            )(x)

            loss = "binary_crossentropy"

            metrics = [
                keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                keras.metrics.Precision(
                    name="precision"
                ),
                keras.metrics.Recall(
                    name="recall"
                ),
                keras.metrics.AUC(
                    name="auc"
                ),
            ]

        else:
            outputs = layers.Dense(
                number_of_classes,
                activation="softmax",
                name="prediction",
            )(x)

            loss = "sparse_categorical_crossentropy"

            metrics = [
                keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ]

    else:
        outputs = layers.Dense(
            1,
            activation="linear",
            name="prediction",
        )(x)

        loss = "mse"

        metrics = [
            keras.metrics.MeanAbsoluteError(
                name="mae"
            )
        ]

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="business_intelligence_ann",
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=loss,
        metrics=metrics,
    )

    return model

In [9]:
# Model Creation and Inspection

ann_model = build_ann_model(
    input_feature_count=X_train_array.shape[1],
    problem_type=problem_type,
    number_of_classes=number_of_classes,
)

ann_model.summary()

Model: "business_intelligence_ann"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,289 (48.00 KB)

 Trainable params: 11,905 (46.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [10]:
#Training Callbacks

if problem_type == "classification":
    monitor_metric = (
        "val_auc"
        if number_of_classes == 2
        else "val_accuracy"
    )

    monitor_mode = "max"

else:
    monitor_metric = "val_loss"
    monitor_mode = "min"


callbacks = [
    keras.callbacks.EarlyStopping(
        monitor=monitor_metric,
        patience=25,
        mode=monitor_mode,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=0.000001,
        verbose=1,
    ),
]

In [11]:
# ANN Training

BATCH_SIZE = min(32, len(X_ann_train))
MAXIMUM_EPOCHS = 300

fit_arguments = {
    "x": X_ann_train,
    "y": y_ann_train,
    "validation_data": (
        X_ann_validation,
        y_ann_validation,
    ),
    "epochs": MAXIMUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "callbacks": callbacks,
    "verbose": 1,
}

if problem_type == "classification":
    fit_arguments["class_weight"] = (
        class_weight_dictionary
    )

training_history = ann_model.fit(
    **fit_arguments
)

print(
    "Completed epochs:",
    len(training_history.history["loss"]),
)

Epoch 1/300
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step - accuracy: 0.5781 - auc: 0.5945 - loss: 0.8850 - precision: 0.6113 - recall: 0.8351 - val_accuracy: 0.8000 - val_auc: 0.8608 - val_loss: 0.6993 - val_precision: 0.8837 - val_recall: 0.7755 - learning_rate: 0.0010
Epoch 2/300
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7000 - auc: 0.7886 - loss: 0.6226 - precision: 0.7311 - recall: 0.7990 - val_accuracy: 0.7250 - val_auc: 0.8779 - val_loss: 0.6938 - val_precision: 0.9655 - val_recall: 0.5714 - learning_rate: 0.0010
Epoch 3/300
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7375 - auc: 0.8202 - loss: 0.5771 - precision: 0.7835 - recall: 0.7835 - val_accuracy: 0.6625 - val_auc: 0.8851 - val_loss: 0.6900 - val_precision: 1.0000 - val_recall: 0.4490 - learning_rate: 0.0010
Epoch 4/300
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7469 - auc: 0.8320 - loss: 0.5505 - precision: 0.8424 - recall: 0.7165 - val_accuracy: 0.6500 - val_auc: 0.8904 - val_loss: 0.6814 - val

In [12]:
# Training History

history_df = pd.DataFrame(
    training_history.history
)

history_df["epoch"] = np.arange(
    1,
    len(history_df) + 1,
)

display(history_df.tail())

loss_figure = px.line(
    history_df,
    x="epoch",
    y=["loss", "val_loss"],
    title="ANN Training and Validation Loss",
)

loss_figure.show()

loss_figure.write_html(
    CHART_DIR / "ann_loss_history.html"
)

,accuracy,auc,loss,precision,recall,val_accuracy,val_auc,val_loss,val_precision,val_recall,learning_rate,epoch
47,0.884375,0.962649,0.282699,0.943503,0.860825,0.7750,0.902238,0.486596,0.969697,0.653061,0.0010,48
48,0.881250,0.966127,0.288166,0.953488,0.845361,0.7750,0.904213,0.475721,0.969697,0.653061,0.0010,49
49,0.893750,0.967436,0.280755,0.949438,0.871134,0.8000,0.902568,0.467956,0.945946,0.714286,0.0010,50
50,0.928125,0.971731,0.260449,0.977654,0.902062,0.7875,0.899276,0.473906,0.944444,0.693878,0.0005,51
51,0.903125,0.976252,0.253703,0.955307,0.881443,0.7750,0.900263,0.473297,0.942857,0.673469,0.0005,52


In [13]:
# Accuracy or MAE History

if problem_type == "classification":
    metric_columns = [
        column
        for column in ["accuracy", "val_accuracy"]
        if column in history_df.columns
    ]

    figure_title = (
        "ANN Training and Validation Accuracy"
    )

else:
    metric_columns = [
        column
        for column in ["mae", "val_mae"]
        if column in history_df.columns
    ]

    figure_title = (
        "ANN Training and Validation MAE"
    )

metric_figure = px.line(
    history_df,
    x="epoch",
    y=metric_columns,
    title=figure_title,
)

metric_figure.show()

metric_figure.write_html(
    CHART_DIR / "ann_metric_history.html"
)

In [14]:
# Generate TEst Predictions

raw_ann_predictions = ann_model.predict(
    X_test_array,
    verbose=0,
)

if problem_type == "classification":
    if number_of_classes == 2:
        prediction_probabilities = (
            raw_ann_predictions.ravel()
        )

        encoded_predictions = (
            prediction_probabilities >= 0.5
        ).astype(int)

    else:
        prediction_probabilities = (
            raw_ann_predictions
        )

        encoded_predictions = np.argmax(
            prediction_probabilities,
            axis=1,
        )

    ann_predictions = label_encoder.inverse_transform(
        encoded_predictions
    )

else:
    scaled_predictions = (
        raw_ann_predictions.reshape(-1, 1)
    )

    ann_predictions = (
        target_scaler.inverse_transform(
            scaled_predictions
        ).ravel()
    )

In [15]:
# ANN Evaluation

ann_metrics = {}

if problem_type == "classification":
    ann_metrics = {
        "accuracy": accuracy_score(
            y_test_original,
            ann_predictions,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test_original,
                ann_predictions,
            ),
        "precision_macro":
            precision_score(
                y_test_original,
                ann_predictions,
                average="macro",
                zero_division=0,
            ),
        "recall_macro":
            recall_score(
                y_test_original,
                ann_predictions,
                average="macro",
                zero_division=0,
            ),
        "f1_macro":
            f1_score(
                y_test_original,
                ann_predictions,
                average="macro",
                zero_division=0,
            ),
        "f1_weighted":
            f1_score(
                y_test_original,
                ann_predictions,
                average="weighted",
                zero_division=0,
            ),
    }

    if number_of_classes == 2:
        ann_metrics["roc_auc"] = roc_auc_score(
            y_test_array,
            prediction_probabilities,
        )

else:
    ann_metrics = {
        "mae": mean_absolute_error(
            y_test_original,
            ann_predictions,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_test_original,
                ann_predictions,
            )
        ),
        "r2": r2_score(
            y_test_original,
            ann_predictions,
        ),
    }

print("ANN test metrics:")

for metric_name, metric_value in ann_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

ANN test metrics:
accuracy: 0.8000
balanced_accuracy: 0.8268
precision_macro: 0.8141
recall_macro: 0.8268
f1_macro: 0.7993
f1_weighted: 0.8019
roc_auc: 0.9113


In [16]:
# Classification Report

if problem_type == "classification":
    print(
        classification_report(
            y_test_original,
            ann_predictions,
            zero_division=0,
        )
    )

    confusion_values = confusion_matrix(
        y_test_original,
        ann_predictions,
    )

    confusion_df = pd.DataFrame(
        confusion_values,
        index=[
            f"Actual {label}"
            for label in label_encoder.classes_
        ],
        columns=[
            f"Predicted {label}"
            for label in label_encoder.classes_
        ],
    )

    display(confusion_df)

              precision    recall  f1-score   support

           0       0.67      0.95      0.79        39
           1       0.96      0.70      0.81        61

    accuracy                           0.80       100
   macro avg       0.81      0.83      0.80       100
weighted avg       0.85      0.80      0.80       100



,Predicted 0,Predicted 1
Actual 0,37,2
Actual 1,18,43


In [17]:
# Compare ANN vs Other Models

supervised_results_file = (
    MODEL_DIR / "test_results.csv"
)

if supervised_results_file.exists():
    supervised_comparison = pd.read_csv(
        supervised_results_file
    )

    if problem_type == "classification":
        ann_comparison_row = pd.DataFrame([{
            "model": "TensorFlow ANN",
            "test_accuracy":
                ann_metrics["accuracy"],
            "test_balanced_accuracy":
                ann_metrics["balanced_accuracy"],
            "test_precision_macro":
                ann_metrics["precision_macro"],
            "test_recall_macro":
                ann_metrics["recall_macro"],
            "test_f1_macro":
                ann_metrics["f1_macro"],
            "test_f1_weighted":
                ann_metrics["f1_weighted"],
            "test_roc_auc":
                ann_metrics.get("roc_auc", np.nan),
        }])

        complete_comparison = pd.concat(
            [
                supervised_comparison,
                ann_comparison_row,
            ],
            ignore_index=True,
        ).sort_values(
            "test_f1_macro",
            ascending=False,
        )

    else:
        ann_comparison_row = pd.DataFrame([{
            "model": "TensorFlow ANN",
            "test_mae": ann_metrics["mae"],
            "test_rmse": ann_metrics["rmse"],
            "test_r2": ann_metrics["r2"],
        }])

        complete_comparison = pd.concat(
            [
                supervised_comparison,
                ann_comparison_row,
            ],
            ignore_index=True,
        ).sort_values(
            "test_r2",
            ascending=False,
        )

    display(complete_comparison.round(4))

,model,test_accuracy,test_balanced_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,test_roc_auc
0,Random Forest,0.90,0.9134,0.8948,0.9134,0.8980,0.9011,0.9361
1,Histogram Gradient Boosting,0.86,0.8714,0.8547,0.8714,0.8572,0.8616,0.9323
2,K-Nearest Neighbours,0.84,0.8411,0.8311,0.8411,0.8346,0.8412,0.9142
3,Gaussian Naive Bayes,0.84,0.8365,0.8309,0.8365,0.8333,0.8407,0.9134
4,Gradient Boosting,0.83,0.8375,0.8233,0.8375,0.8261,0.8318,0.9361
5,ANN MLP,0.83,0.8329,0.8214,0.8329,0.8249,0.8315,0.9092
6,Support Vector Machine,0.82,0.8386,0.8223,0.8386,0.8182,0.8222,0.8949
7,Logistic Regression,0.82,0.8293,0.8145,0.8293,0.8164,0.8221,0.9071
8,Extra Trees,0.82,0.8062,0.8125,0.8062,0.8090,0.8191,0.9151
10,TensorFlow ANN,0.80,0.8268,0.8141,0.8268,0.7993,0.8019,0.9113


In [18]:
# Save Predictions

ann_predictions_df = pd.DataFrame({
    "actual": y_test_original.reset_index(
        drop=True
    ),
    "ann_prediction": ann_predictions,
})

if (
    problem_type == "classification"
    and number_of_classes == 2
):
    ann_predictions_df[
        "probability_class_1"
    ] = prediction_probabilities

ann_predictions_df.to_csv(
    PREDICTION_DIR / "ann_test_predictions.csv",
    index=False,
)

display(ann_predictions_df.head(10))

,actual,ann_prediction,probability_class_1
0,1,0,0.366835
1,1,0,0.242257
2,1,1,0.695815
3,0,0,0.080672
4,0,0,0.216050
5,1,0,0.298873
6,0,0,0.152068
7,1,0,0.306899
8,1,1,0.907418
9,1,1,0.957658


In [19]:
# Save ANN Artifacts

ANN_MODEL_PATH = MODEL_DIR / "tensorflow_ann.keras"

ann_model.save(ANN_MODEL_PATH)

history_df.to_csv(
    MODEL_DIR / "ann_training_history.csv",
    index=False,
)

with open(
    MODEL_DIR / "ann_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            key: float(value)
            for key, value in ann_metrics.items()
        },
        file,
        indent=4,
    )

if label_encoder is not None:
    joblib.dump(
        label_encoder,
        MODEL_DIR / "ann_label_encoder.joblib",
    )

if target_scaler is not None:
    joblib.dump(
        target_scaler,
        MODEL_DIR / "ann_target_scaler.joblib",
    )

ann_metadata = {
    "problem_type": problem_type,
    "target_column": TARGET_COLUMN,
    "input_feature_count": int(
        X_train_array.shape[1]
    ),
    "input_features": X_train.columns.tolist(),
    "completed_epochs": int(
        len(history_df)
    ),
    "batch_size": int(BATCH_SIZE),
    "maximum_epochs": int(MAXIMUM_EPOCHS),
    "model_path": str(ANN_MODEL_PATH),
}

with open(
    MODEL_DIR / "ann_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        ann_metadata,
        file,
        indent=4,
    )

print("ANN artifacts saved successfully.")

ANN artifacts saved successfully.


In [20]:
# Verify Saved outputs

required_files = [
    MODEL_DIR / "tensorflow_ann.keras",
    MODEL_DIR / "ann_training_history.csv",
    MODEL_DIR / "ann_metrics.json",
    MODEL_DIR / "ann_metadata.json",
    PREDICTION_DIR / "ann_test_predictions.csv",
]

if problem_type == "classification":
    required_files.append(
        MODEL_DIR / "ann_label_encoder.joblib"
    )

if problem_type == "regression":
    required_files.append(
        MODEL_DIR / "ann_target_scaler.joblib"
    )

for path in required_files:
    print(f"{path.name}: {path.exists()}")

tensorflow_ann.keras: True
ann_training_history.csv: True
ann_metrics.json: True
ann_metadata.json: True
ann_test_predictions.csv: True
ann_label_encoder.joblib: True
